# ML-04 — Ranking Signal Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgrivero/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This notebook defines and verifies my lane on a mid-panel month. It uses pseudonymized warehouse data only; IDs stay context fields and no raw IDs are displayed.

In [1]:
import os
from pathlib import Path
import getpass
import duckdb
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def load_hf_token():
    token = os.environ.get('HF_TOKEN')
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            return token
    except (ImportError, ModuleNotFoundError, KeyError):
        pass
    cached = Path.home() / '.cache' / 'huggingface' / 'token'
    if cached.exists():
        return cached.read_text(encoding='utf-8').strip()
    return getpass.getpass('Hugging Face READ token (input hidden): ')

con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [load_hf_token()])
REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('Connected to the warehouse; token was loaded without being displayed.')

Connected to the warehouse; token was loaded without being displayed.


## 1. Contract: unit of analysis + time window

1. **One row means:** one pseudonymized content page for one pseudonymized client, aggregated from daily observations. The raw fact table is one row per date × client × page; Query 3 turns it into my page-level modeling grain.
2. **Table:** `fact_content_daily_performance`, specifically its `month=2026-03` partition.
3. **Time window:** 2026-03-01 through 2026-03-31. The decision moment is 2026-04-01, after March has closed. June remains sealed as the natural final test month.
4. **What I rank:** pages by predicted March GA4 engagement rate, `SUM(ga4_engaged_sessions) / SUM(ga4_sessions)`. This is an observed proxy for engagement, not a causal measure of content quality. Low predicted scores would go nearer the top of a review queue.
5. **Deliberate exclusion:** pseudonymous IDs are context for grouping and splitting only, never model features; I also exclude product flags and all direct ingredients/transformations of the label.

## 2. Field roles

- **Features (exactly five):** `log_gsc_impressions`, `gsc_ctr`, `gsc_avg_position`, `log_ga4_sessions`, `pageviews_per_session`.
- **Label/proxy:** `engagement_rate`; it is calculated from `ga4_engaged_sessions` and `ga4_sessions`.
- **Context:** `client_hash_id` and `content_hash_id` enforce the grain and support a client-grouped split; `report_date` defines the window.
- **Excluded:** `ga4_engaged_sessions` and any engagement-rate transformation are label-derived; `client_has_*`, `*_data_available`, and publishing/optimization flags are operational/product decisions rather than page signals. The availability flag is used only as an eligibility filter.

## 3. Three verification queries, five features, and the trap

### Query 1 of 3 — raw-table grain

Zero returned rows means no duplicate date × client × page keys were found.

In [2]:
query_1 = f'''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM {MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
'''
grain_violations = con.sql(query_1).df()
print(f'Grain violations returned: {len(grain_violations)}')
grain_violations

Grain violations returned: 0


,report_date,client_hash_id,content_hash_id,row_count


### Query 2 of 3 — slice count and date span

This verifies the size and exact boundaries of the March slice.

In [3]:
query_2 = f'''
SELECT COUNT(*) AS daily_rows,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date,
       COUNT(DISTINCT content_hash_id) AS distinct_pages
FROM {MARCH}
'''
slice_check = con.sql(query_2).df()
slice_check

,daily_rows,first_date,last_date,distinct_pages
0,9841378,2026-03-01,2026-03-31,331437


### Query 3 of 3 — availability filter and page-level feature frame

The literal `IS TRUE` prevents pre-GA4 zero-filled rows from masquerading as measured zero engagement. This query reports the survival counts and builds the page-level frame in one pass. Pages with zero measured sessions cannot have a defined engagement-rate label and are omitted.

In [4]:
query_3 = f'''
WITH counts AS (
    SELECT COUNT(*) AS source_daily_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS available_daily_rows
    FROM {MARCH}
), page_month AS (
    SELECT client_hash_id, content_hash_id,
           LN(1 + SUM(gsc_impressions)) AS log_gsc_impressions,
           SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS gsc_ctr,
           SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,
           LN(1 + SUM(ga4_sessions)) AS log_ga4_sessions,
           SUM(ga4_pageviews)::DOUBLE / NULLIF(SUM(ga4_sessions), 0) AS pageviews_per_session,
           SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(ga4_sessions) > 0
)
SELECT p.*, c.source_daily_rows, c.available_daily_rows
FROM page_month p CROSS JOIN counts c
'''
raw_frame = con.sql(query_3).df()
source_rows = int(raw_frame.pop('source_daily_rows').iloc[0])
available_rows = int(raw_frame.pop('available_daily_rows').iloc[0])
print(f'Daily rows before availability filter: {source_rows:,}')
print(f'Daily rows surviving ga4_data_available IS TRUE: {available_rows:,} ({available_rows/source_rows:.2%})')
print(f'Page rows with a defined label: {len(raw_frame):,}')
raw_frame.drop(columns=['client_hash_id', 'content_hash_id']).head()

Daily rows before availability filter: 9,841,378
Daily rows surviving ga4_data_available IS TRUE: 413,966 (4.21%)
Page rows with a defined label: 90,237


,log_gsc_impressions,gsc_ctr,gsc_avg_position,log_ga4_sessions,pageviews_per_session,engagement_rate
0,0.000000,NaN,NaN,1.098612,1.000000,0.000000
1,8.352790,0.000236,27.449422,6.001415,1.004963,0.000000
2,8.662159,0.003115,37.866932,3.871201,1.212766,0.000000
3,6.807935,0.005531,13.963496,2.995732,1.315789,0.000000
4,5.068904,0.006329,18.000000,2.079442,1.142857,0.285714


### Why each feature is available at the decision moment

The decision moment is **2026-04-01**, after the March reporting window has closed.

- `log_gsc_impressions` — knowable then because March GSC impression counts have already landed; the log only rescales them.
- `gsc_ctr` — knowable then because both March clicks and impressions are already observed.
- `gsc_avg_position` — knowable then because March impression-weighted position totals are already observed.
- `log_ga4_sessions` — knowable then because March GA4 sessions have already landed; the log only rescales them.
- `pageviews_per_session` — knowable then because March pageviews and sessions are already observed.

These are valid for a retrospective April 1 review queue. They are not evidence that the model can forecast a later month's engagement.

In [5]:
feature_cols = [
    'log_gsc_impressions', 'gsc_ctr', 'gsc_avg_position',
    'log_ga4_sessions', 'pageviews_per_session'
]
model_frame = raw_frame.dropna(subset=feature_cols + ['engagement_rate']).copy()
assert len(feature_cols) == 5
assert model_frame[['client_hash_id', 'content_hash_id']].duplicated().sum() == 0
print(f'Feature frame: {len(model_frame):,} page rows × {len(feature_cols)} features')
model_frame[feature_cols + ['engagement_rate']].head()

Feature frame: 63,695 page rows × 5 features


,log_gsc_impressions,gsc_ctr,gsc_avg_position,log_ga4_sessions,pageviews_per_session,engagement_rate
1,8.352790,0.000236,27.449422,6.001415,1.004963,0.000000
2,8.662159,0.003115,37.866932,3.871201,1.212766,0.000000
3,6.807935,0.005531,13.963496,2.995732,1.315789,0.000000
4,5.068904,0.006329,18.000000,2.079442,1.142857,0.285714
5,7.016610,0.006284,19.197487,3.610918,1.055556,0.000000


### The deliberate leakage trap

I deliberately copy the label into `engagement_rate_leak`. A client-grouped holdout keeps pages from the same client on one side of the split, but no split can rescue a feature that directly contains the answer. I compare Spearman rank correlation because this lane produces a ranked queue.

In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(model_frame, groups=model_frame['client_hash_id']))
y = model_frame['engagement_rate']

def rank_score(columns):
    model = make_pipeline(StandardScaler(), LinearRegression())
    model.fit(model_frame.iloc[train_idx][columns], y.iloc[train_idx])
    predictions = model.predict(model_frame.iloc[test_idx][columns])
    return spearmanr(y.iloc[test_idx], predictions).statistic

honest_rho = rank_score(feature_cols)
model_frame['engagement_rate_leak'] = model_frame['engagement_rate']  # deliberate leak
leaky_rho = rank_score(['engagement_rate_leak'])
print(f'Honest held-out Spearman rho: {honest_rho:.3f}')
print(f'With label-derived leak:       {leaky_rho:.3f}')

Honest held-out Spearman rho: 0.143
With label-derived leak:       1.000


In [7]:
model_frame = model_frame.drop(columns='engagement_rate_leak')
assert 'engagement_rate_leak' not in model_frame.columns
assert feature_cols == list(model_frame[feature_cols].columns)
final_score = honest_rho
print(f'Leak deleted. Honest score retained: rho = {final_score:.3f}')

Leak deleted. Honest score retained: rho = 0.143


## 4. Named limitation

**GA4 coverage selection:** only 413,966 of the 9,841,378 March daily rows have `ga4_data_available IS TRUE`, and pages also need at least one measured session to receive this label. The resulting queue therefore describes the GA4-observed subset, not every warehouse page. Because client histories are unbalanced and GA4 starts at different times, the missingness is structural rather than random. Also, this same-month exercise measures association for an April 1 review queue; it does not yet demonstrate future-month forecasting.

## 5. Self-check

- [x] Five contract answers are stated in plain words.
- [x] Exactly three verification queries are executed and their outputs are visible.
- [x] Availability is filtered with `IS TRUE`, with before/after counts shown.
- [x] The frame has exactly five features and an available-when line for each.
- [x] One label-derived feature is deliberately added, scored, then deleted.
- [x] One limitation is named.
- [x] No token, client name, URL, raw identifier, or private query result is displayed.
- [x] June 2026 remains sealed as the final test month.